Establish connection to Oracle Database using SQLAlchemy with specified credentials

In [ ]:
import pandas as pd #Imports pandas library for data manipulation and analysis.
from sqlalchemy import create_engine    #Imports SQLAlchemy function to create a connection engine for interacting with databases.

#Define Oracle database connection parameters including username, password, host, port, and service name.
user = "user"
password = "pwd"
host = "host"
port = "port"
service = "ORCL"

#Creates a SQLAlchemy engine to establish a connection with the Oracle database using provided credentials.
engine = create_engine(
    f"oracle+oracledb://{user}:{password}@{host}:{port}/?service_name={service}"
)

Execute SQL query to retrieve data from the Oracle table and load it into a pandas DataFrame for further analysis.

In [31]:
query = """
SELECT *
FROM SAPTEST.PERF_FACT_MONTHLY_SNAPSHOT
"""

df = pd.read_sql(query, engine)

Display the first few rows of the DataFrame to quickly inspect the structure and contents of the loaded data.

In [32]:
df.head()

,snapshot_month,emp_id,gender,tenure_yy,age_yy,cadre,grade,unit,location,doj,...,prod_3m_slope,prod_volatility,prod_risk_flag,overtime_3m_avg,absence_3m_avg,cons_overtime_months,leave_spike_flag,burnout_risk_flag,pay_growth_trend,comp_stag_flag
0,2021-07-01,5E3C217687,1,36.24,57,1,0,1010,2,1985-05-03,...,NaN,6.60,0,8.27,1.00,0,0,0,0.0,1
1,2021-08-01,5E3C217687,1,36.33,57,1,0,1010,2,1985-05-03,...,2.07,5.90,0,6.90,1.00,0,0,0,0.0,1
2,2021-09-01,5E3C217687,1,36.41,57,1,0,1010,2,1985-05-03,...,-0.32,5.14,0,8.20,0.67,0,0,0,0.0,1
3,2021-10-01,5E3C217687,1,36.49,57,1,0,1010,2,1985-05-03,...,-0.28,4.60,0,8.00,0.67,0,0,0,0.0,1
4,2021-11-01,5E3C217687,1,36.57,57,1,0,1010,2,1985-05-03,...,-1.64,3.57,0,9.63,1.00,0,1,1,0.0,1


List all column names in the DataFrame to understand the available fields in the dataset.

In [33]:
df.columns

Index(['snapshot_month', 'emp_id', 'gender', 'tenure_yy', 'age_yy', 'cadre',
       'grade', 'unit', 'location', 'doj', 'dor', 'yy_since_prom',
       'stagflation', 'career_velocity', 'tasks_assigned', 'tasks_completed',
       'prod_rate', 'timeliness', 'error_rate', 'work_hours', 'utilise_rate',
       'produc_hours', 'high_prod_flag', 'rating', 'kpi', 'goal_perc',
       'mgr_rating', 'potential', 'perf_change', 'basic', 'bonus', 'incentive',
       'total', 'compa_ratio', 'sal_growth', 'engage_score', 'burn_score',
       'well_score', 'manager_score', 'culture_score', 'workload_score',
       'working_days', 'absence_days', 'sick_leaves', 'casual_leaves',
       'late_logins', 'early_exits', 'overtime_hh', 'train_hh_6m',
       'avg_train_score_6m', 'cert_count', 'prod_3m_avg', 'prod_3m_slope',
       'prod_volatility', 'prod_risk_flag', 'overtime_3m_avg',
       'absence_3m_avg', 'cons_overtime_months', 'leave_spike_flag',
       'burnout_risk_flag', 'pay_growth_trend', 'comp_st

Calculate the total number of missing (null) values in each column to assess data completeness.

In [34]:
na_counts = df.isna().sum()

# Show only columns where NA count > 0
na_counts = na_counts[na_counts > 0]
na_counts

yy_since_prom          7137
career_velocity           4
rating                10196
kpi                   10196
goal_perc             10196
mgr_rating            10196
potential             10196
perf_change           10196
engage_score          24089
burn_score            24089
well_score            24089
manager_score         24089
culture_score         24089
workload_score        24089
train_hh_6m           92655
avg_train_score_6m    92655
prod_3m_slope         39249
dtype: int64

Perform data cleaning by handling missing values using defaults, medians, and group-wise imputations to ensure completeness and consistency of features for analysis.

In [35]:
df2 = df.copy()
df2['yy_since_prom'] = df2['yy_since_prom'].fillna(df['tenure_yy'])
df2['rating'] = df2.groupby('grade')['rating'].transform(lambda x: x.fillna(x.median()))
df2['kpi'] = df2.groupby('grade')['kpi'].transform(lambda x: x.fillna(x.median()))
df2['goal_perc'] = df2.groupby('grade')['goal_perc'].transform(lambda x: x.fillna(x.median()))
df2['mgr_rating'] = df2.groupby('grade')['mgr_rating'].transform(lambda x: x.fillna(x.median()))
df2['potential'] = df2.groupby('grade')['potential'].transform(lambda x: x.fillna(x.median()))
df2['perf_change'] = df2['yy_since_prom'].fillna(0)
df2['train_hh_6m'] = df2['train_hh_6m'].fillna(0)
df2['avg_train_score_6m'] = df2['avg_train_score_6m'].fillna(0)
df2['prod_3m_slope'] = df2['prod_3m_slope'].fillna(0)
df2['career_velocity'] = df2['career_velocity'].fillna(0)
df2['engage_score'] = df2.groupby('unit')['engage_score'].transform(lambda x: x.fillna(x.median()))
df2['burn_score'] = df2.groupby('unit')['burn_score'].transform(lambda x: x.fillna(x.median()))
df2['well_score'] = df2.groupby('unit')['well_score'].transform(lambda x: x.fillna(x.median()))
df2['manager_score'] = df2.groupby('unit')['manager_score'].transform(lambda x: x.fillna(x.median()))
df2['culture_score'] = df2.groupby('unit')['culture_score'].transform(lambda x: x.fillna(x.median()))
df2['workload_score'] = df2.groupby('unit')['workload_score'].transform(lambda x: x.fillna(x.median()))

In [36]:
na_counts = df2.isna().sum()
# Show only columns where NA count > 0
na_counts = na_counts[na_counts > 0]
na_counts

Series([], dtype: int64)

Converting Colum Headers to Upper Case

In [37]:
df2.columns = df2.columns.str.upper()

Fetch Latest Dataframe (Snapshot Date = 2026-05-01) & Training Dataframe (Snapshot Date between 01-05-2021 to 01-04-2026)

In [38]:
latest_snapshot = pd.Timestamp("2026-05-01")

df_latest = df2[df2["SNAPSHOT_MONTH"] == latest_snapshot].copy()

df_train = df2[(df2["SNAPSHOT_MONTH"] <= "2026-04-01") & (df2["SNAPSHOT_MONTH"] >= "2021-05-01")].copy()

Define target variable for model training.

In [39]:
y = df_train["PROD_RISK_FLAG"]

Handle non-numeric entries in Grade Field.

In [40]:
df_train['GRADE'] = df_train['GRADE'].str.replace("T","9")

Identify highly correlated features (>0.8) to remove multicollinearity in the dataset.

In [42]:
import pandas as pd
import numpy as np

# correlation matrix
corr_matrix = df_train.drop(columns="EMP_ID",axis=1).corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# find columns with correlation > threshold
threshold = 0.8

to_drop = [
    column for column in upper.columns
    if any(upper[column] > threshold)
]

print("Columns to drop:", to_drop)

Columns to drop: ['AGE_YY', 'GRADE', 'DOJ', 'DOR', 'TASKS_COMPLETED', 'TIMELINESS', 'PRODUC_HOURS', 'PERF_CHANGE', 'TOTAL', 'CASUAL_LEAVES', 'PROD_3M_AVG', 'OVERTIME_3M_AVG', 'BURNOUT_RISK_FLAG']


In [ ]:
df3 = df2.drop(columns=['AGE_YY', 'GRADE', 'DOJ', 'DOR', 'TASKS_COMPLETED', 'TIMELINESS', 'PRODUC_HOURS', 'PERF_CHANGE', 'TOTAL', 'CASUAL_LEAVES', 'PROD_3M_AVG', 'OVERTIME_3M_AVG', 'BURNOUT_RISK_FLAG'])

Remove highly correlated columns & columns which can lead to data leakage for improving accuracy of ML model and prediction.

In [43]:
drop_cols = [
    "EMP_ID", "SNAPSHOT_MONTH",
    "PROD_RISK_FLAG", "HIGH_PROD_FLAG",'BURNOUT_RISK_FLAG',
    "PROD_RATE", "ERROR_RATE", "PROD_VOLATILITY",'PROD_3M_AVG',
    "BASIC", "BONUS", "INCENTIVE",'TOTAL',
    "GENDER", "UNIT", "LOCATION", "CADRE", 'AGE_YY', 'GRADE', 'DOJ', 'DOR', 
    'TASKS_COMPLETED', 'TIMELINESS', 'PRODUC_HOURS', 'PERF_CHANGE', 'CASUAL_LEAVES', 'OVERTIME_3M_AVG'
]

X = df_train.drop(columns=drop_cols, errors="ignore")
X_latest = df_latest.drop(columns=drop_cols, errors="ignore")

Import libraries for data handling, database access, model training, and evaluation.

In [44]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score

Split data into training and test sets while preserving class distribution (Train = 75% / Test = 25%). 

In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42, #Controls randomness
    stratify=y #Ensures class balance is preserved
)

Create a pipeline with scaling and logistic regression model for balanced classification.

In [46]:
pipe = Pipeline([
    ("scaler", StandardScaler()), #standardizes features
    ("model", LogisticRegression(  #create Logistic Regression Model
        class_weight="balanced",
        max_iter=2000,
        C=0.5
    ))
])

Train the model pipeline using the training data.

In [47]:
pipe.fit(X_train, y_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.5


Generate predicted probabilities for test data.

In [48]:
pred_test = pipe.predict_proba(X_test)[:,1]

Calculate ROC AUC score to evaluate model performance.

In [49]:
auc = roc_auc_score(y_test, pred_test)

print("ROC AUC Score:", auc)

ROC AUC Score: 0.8181647401051205


Generate predicted attrition probabilities for the latest dataset.

In [50]:
prob_latest = pipe.predict_proba(X_latest)[:,1]
df_latest["PROD_RISK_PROB"] = prob_latest.round(6)

Define database credentials and create Oracle connection engine for data access.

In [ ]:
user = "user"
password = "pwd"
host = "host"
port = "port"
service = "ORCL"

engine = create_engine(
    f"oracle+oracledb://{user}:{password}@{host}:{port}/?service_name={service}"
)

In [52]:
df_latest.columns

Index(['SNAPSHOT_MONTH', 'EMP_ID', 'GENDER', 'TENURE_YY', 'AGE_YY', 'CADRE',
       'GRADE', 'UNIT', 'LOCATION', 'DOJ', 'DOR', 'YY_SINCE_PROM',
       'STAGFLATION', 'CAREER_VELOCITY', 'TASKS_ASSIGNED', 'TASKS_COMPLETED',
       'PROD_RATE', 'TIMELINESS', 'ERROR_RATE', 'WORK_HOURS', 'UTILISE_RATE',
       'PRODUC_HOURS', 'HIGH_PROD_FLAG', 'RATING', 'KPI', 'GOAL_PERC',
       'MGR_RATING', 'POTENTIAL', 'PERF_CHANGE', 'BASIC', 'BONUS', 'INCENTIVE',
       'TOTAL', 'COMPA_RATIO', 'SAL_GROWTH', 'ENGAGE_SCORE', 'BURN_SCORE',
       'WELL_SCORE', 'MANAGER_SCORE', 'CULTURE_SCORE', 'WORKLOAD_SCORE',
       'WORKING_DAYS', 'ABSENCE_DAYS', 'SICK_LEAVES', 'CASUAL_LEAVES',
       'LATE_LOGINS', 'EARLY_EXITS', 'OVERTIME_HH', 'TRAIN_HH_6M',
       'AVG_TRAIN_SCORE_6M', 'CERT_COUNT', 'PROD_3M_AVG', 'PROD_3M_SLOPE',
       'PROD_VOLATILITY', 'PROD_RISK_FLAG', 'OVERTIME_3M_AVG',
       'ABSENCE_3M_AVG', 'CONS_OVERTIME_MONTHS', 'LEAVE_SPIKE_FLAG',
       'BURNOUT_RISK_FLAG', 'PAY_GROWTH_TREND', 'COMP_ST

Normlaise columns to be used for calculation of Future Readiness Index of employees.

In [53]:
from sklearn.preprocessing import MinMaxScaler

cols = [
    'TRAIN_HH_6M',
    'ENGAGE_SCORE',
    'CAREER_VELOCITY',
    'PROD_RATE',
    'PROD_RISK_PROB'
]

scaler = MinMaxScaler()

scaled = scaler.fit_transform(df_latest[cols])

In [54]:
scaled_df = pd.DataFrame(
    scaled,
    columns=cols
)

In [55]:
scaled_df.describe

<bound method NDFrame.describe of        TRAIN_HH_6M  ENGAGE_SCORE  CAREER_VELOCITY  PROD_RATE  PROD_RISK_PROB
0         0.037833      0.709239         0.354430   0.524093        0.770942
1         0.038300      0.707880         0.240506   0.731088        0.323483
2         0.309201      0.707880         0.316456   0.803109        0.183653
3         0.187763      0.675272         0.177215   0.502591        0.722810
4         0.000000      0.675272         0.316456   0.620984        0.328803
...            ...           ...              ...        ...             ...
11087     0.272770      0.705163         0.569620   1.000000        0.312933
11088     0.475479      0.705163         0.569620   0.350777        0.527680
11089     0.000000      0.683424         0.569620   0.511658        0.641474
11090     0.286315      0.729620         0.569620   0.414508        0.400997
11091     0.143858      0.729620         0.658228   0.869689        0.418092

[11092 rows x 5 columns]>

In [56]:
scaled_df['READINESS_INDEX'] = (

    0.25 * scaled_df['ENGAGE_SCORE']

     + 0.25 * scaled_df['PROD_RATE']

    + 0.20 * scaled_df['TRAIN_HH_6M']

    + 0.15 * scaled_df['CAREER_VELOCITY']

    + 0.15 * (1 - scaled_df['PROD_RISK_PROB'])

) * 100

Appending Readiness Index column to df_latest dataframe.

In [57]:
df_latest['READINESS_INDEX'] = scaled_df['READINESS_INDEX'].values

Write predicted results to Oracle table in batches for efficient database storage.

In [59]:
df_latest.to_sql(
    name="PERF_PROD_RISK_PROBAB",
    schema="SAPTEST",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=5000
)

C:\Users\asad\AppData\Local\Temp\ipykernel_7456\2315138634.py:1: UserWarning: The provided table name 'PERF_PROD_RISK_PROBAB' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_latest.to_sql(


-3

Extract model coefficients and map them to features for interpretability.

In [60]:
import pandas as pd
import numpy as np

# Extract trained logistic regression model
model = pipe.named_steps["model"]

# Create dataframe of coefficients
coef_df = pd.DataFrame({
    "FEATURE": X.columns,
    "COEFF": model.coef_[0].round(8)
})

print(coef_df.head())

           FEATURE     COEFF
0        TENURE_YY -0.109629
1    YY_SINCE_PROM  0.009985
2      STAGFLATION -0.030839
3  CAREER_VELOCITY -0.013277
4   TASKS_ASSIGNED  0.002027


Save feature coefficients to Oracle table for attrition driver analysis.

In [61]:
coef_df.to_sql(
    name="PERF_PROD_RISK_COEFF",
    schema="SAPTEST",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=5000
)

C:\Users\asad\AppData\Local\Temp\ipykernel_7456\1210651534.py:1: UserWarning: The provided table name 'PERF_PROD_RISK_COEFF' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  coef_df.to_sql(


-1